# 6.4 詞表要多大？


超市品項更多，可能讓常見需求一次買齊，也需要更多貨架。tokenizer的字表更大，也可能把長片段直接收成一個token，縮短模型序列；同時每個token要有輸入特徵與候選分數，模型表格就更大。不能只說「詞表越大越好」或「越小越省」，要把序列長度與參數成本一起量。

[6.3](https://birdhackor.github.io/tiny-perceptron-vlm/6.3.html) 用完整byte起點加少量合併，大字表預算則允許更多常見組合。假設每token有D=32特徵，V個token的embedding需要V×32個數，原理見 [2.1](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html)。若 [4.6](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html) 的輸出表與輸入不共享，還要另一張同樣V×32的權重；注意力與FFN則不會只因V變十倍也自動變十倍。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
width = 32
for vocab in [300, 1000, 3000]:
    embedding = vocab * width
    input_and_output = 2 * embedding
    print("詞表", vocab, "輸入參數", embedding, "輸入FP32 bytes", embedding * 4, "不共享輸入輸出參數", input_and_output)

輸入三個詞表預算，程序只計表格格數。V300時embedding9600、FP32數據38400bytes；V1000是32000與128000；V3000是96000與384000。不共享輸入輸出則分別19200、64000、192000格。FP32四bytes的約定見 [5.9](https://birdhackor.github.io/tiny-perceptron-vlm/5.9.html)，這些數字不含訓練歷史、梯度或其他層，所以不是整機記憶體預算。

成本另一邊是實際T。更多合併如果剛好覆蓋當前文本，token數可能減少，注意力T×T便更省。但字表大不保證這份驗證中文一定切得短：如果訓練tokenizer的資料主要是英文，新增片段可能幫不上中文。應拿同一份未參與字表訓練的原文，記錄實際token數、還原是否正確與模型品質，再看預算。

太罕見的大token也可能只有極少訓練例子，雖然縮短某條句子，卻讓模型難以學好那份輸入輸出參數。反之，小片段需要更多步才能組成意思，卻可以在不同詞中共享資料。這個取捨和語料分佈、模型寬度、訓練token預算有關，沒有脫離任務的唯一最佳V。

比較字表大小時還需讓模型與各自tokenizer一致。相同的ID27可能代表不同片段，不能只改config的vocab_size就沿用舊錶的字義。換字表後要重新建立或訓練匹配模型，保存規則版本，纔是真正的對照。

用[6.3的那19篇驗證片段](https://birdhackor.github.io/tiny-perceptron-vlm/6.3.html)實際量一次，就能看到這個取捨。逐byte工具有256種原文byte，加八個結構項，共264項；BPE則共512項。同一批原文，前者需3,894個普通文字token，後者只需2,112個，平均每篇約205格降至111格；這裡尚未加開始與結束標記。但兩個同樣寬度32、一層的模型，參數卻由41,824個增加至57,696個。片段較短與模型較大同時發生，不能只挑其中一邊就說更省；下一節再把預測品質放進來。

練習只把width從32改成64，先預測每項格數與bytes都翻倍，再執行核對；詞表V不變，所以你增加的是每token描述寬度。接着把V300改600，第一項也翻倍，但這次改變的是可用token種類，兩種成本變化相似，資料含義卻不同。
